# Vérification LLM des codes post-coordonnés CIM-11 — V2 (+ validation anti-hallucination)

**Améliorations par rapport à la V1** :
- Prompt renforcé : Mistral ne doit utiliser QUE les codes d'extension fournis
- Limite abaissée à 100 codes par code racine (au lieu de 200) pour meilleure qualité
- Règles de chaque axe explicitées (NotAllowed vs AllowAlways)
- Dossier de sortie dédié : `verif_postcoord/V2/`
- Reconstruction automatique des libellés (racine + extensions) en post-traitement, sans les redemander à Mistral
- **Validation anti-hallucination** : chaque code renvoyé par Mistral est vérifié contre les référentiels locaux ; les codes inexistants (ex : `XA13Z`) sont écartés et journalisés

**Entrées** (fichiers de référence déjà produits) :
- `codes_cim11.csv` : libellés des codes racines
- `axes_par_code.csv` : axes de chaque code racine avec taille et règle
- `codes_extension_libelles.csv` : libellés des codes d'extension
- `cache_get_url.pkl` : cache API pour reconstruire les axes détaillés

**Sorties** :
- `codes_postcoord_realistes_v2.csv` (une ligne par code post-coordonné, avec libellé du code racine et libellés des extensions)
- `codes_hallucines_v2.csv` : journal des codes inventés par Mistral et écartés

**Ordre d'exécution** : les cellules sont organisées dans l'ordre logique d'exécution (toutes les fonctions sont définies avant d'être utilisées, y compris pour la cellule de test).

## 1. Imports et configuration

In [1]:
# Installation du SDK Mistral (version figée pour compatibilité avec les autres notebooks)
!pip install mistralai==1.2.0 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.4/254.4 kB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 27.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 8.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
holidays 0.100 requires python-dateutil<3,>=2.9.0.post0, but you have python-dateutil 2.8.2 which is incompatible.
google-genai 2.11.0 requires httpx<1.0.0,>=0.28.1, but you have httpx 0.27.2 which is incompatible.
firebase-admin 6.9.0 requires httpx[http2]==0.28.1, but you have httpx 0.27.2 which is incompatible.


In [2]:
# ── Bibliothèques standard ──────────────────────────────────────────────
import json
import os
import re
import time
import pickle
import pandas as pd
from io import BytesIO
from tqdm.auto import tqdm

# ── Montage du Google Drive (accès aux fichiers de référence et de sortie) ─
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── Chemins ────────────────────────────────────────────────────────────
PATH_BASE_API  = "/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/"
PATH_REFERENCE = PATH_BASE_API + "postcoord_reference/"
PATH_VERIF     = PATH_BASE_API + "verif_postcoord/"

# Fichiers d'entrée (produits par les notebooks précédents)
PATH_CODES_CSV      = PATH_REFERENCE + "data/codes_cim11.csv"
PATH_AXES_CSV       = PATH_REFERENCE + "data/axes_par_code.csv"
PATH_EXTENSIONS_CSV = PATH_VERIF + "data/codes_extension_libelles.csv"
PATH_CACHE = PATH_REFERENCE + "cache/cache_get_url.pkl"

# Fichiers de sortie
PATH_BATCH_DIR = PATH_VERIF + "batches/"
PATH_RESULTAT  = PATH_VERIF + "data/codes_postcoord_realistes_v2.csv"

os.makedirs(PATH_BATCH_DIR, exist_ok=True)
os.makedirs(PATH_VERIF + "data/", exist_ok=True)

# ── Configuration Mistral ──────────────────────────────────────────────
import sys
sys.path.append("/content/drive/MyDrive/Colab_Notebooks/Serenic_M/Fine-Tuning/")
from config import api_key

from mistralai import File, Mistral

MODEL_MISTRAL      = "mistral-large-latest"
N_CODES_PAR_BATCH  = 100
N_CODES_POSTCOORD  = 100   # nombre de codes post-coordonnés demandés par code racine

client_mistral = Mistral(api_key=api_key)

print("Configuration OK")

Configuration OK


## 2. Chargement des fichiers de référence

In [4]:
# Codes racines avec libellés
df_codes = pd.read_csv(PATH_CODES_CSV)
dict_libelle_racine = dict(zip(df_codes['code'], df_codes['libelle']))
dict_uri = dict(zip(df_codes['code'], df_codes['uri'])) # Moved here
print(f"Codes racines chargés : {len(df_codes)}")

# Axes par code
df_axes = pd.read_csv(PATH_AXES_CSV)
print(f"Axes chargés : {len(df_axes)} ({df_axes['code'].nunique()} codes distincts)")

# Libellés des codes d'extension
df_extensions = pd.read_csv(PATH_EXTENSIONS_CSV)
dict_libelle_ext = dict(zip(df_extensions['code'], df_extensions['libelle']))
print(f"Codes d'extension chargés : {len(df_extensions)}")

# Cache des réponses API OMS
with open(PATH_CACHE, 'rb') as f:
    cache_url = pickle.load(f)
print(f"Cache rechargé : {len(cache_url)} URI en mémoire")

Codes racines chargés : 34663
Axes chargés : 21678 (10287 codes distincts)
Codes d'extension chargés : 16842
Cache rechargé : 68855 URI en mémoire


### Référentiels de validation (anti-hallucination)

Mistral invente parfois des codes d'extension inexistants (ex : `XA13Z`, absent du référentiel OMS).
On construit ici deux ensembles de codes réels, à partir des dictionnaires déjà chargés ci-dessus :

- `CODES_RACINES_VALIDES` : tous les codes racines connus (`codes_cim11.csv`)
- `CODES_EXTENSIONS_VALIDES` : tous les codes d'extension connus (`codes_extension_libelles.csv`)

Ces ensembles servent à valider chaque code renvoyé par Mistral, **sans aucun appel API**.

In [5]:
# ── Référentiels de validation (anti-hallucination) ────────────────────
# Ensembles construits à partir des dictionnaires chargés dans la cellule précédente
CODES_RACINES_VALIDES    = set(dict_libelle_racine.keys())
CODES_EXTENSIONS_VALIDES = set(dict_libelle_ext.keys())

print(f"Codes racines valides    : {len(CODES_RACINES_VALIDES)}")
print(f"Codes extensions valides : {len(CODES_EXTENSIONS_VALIDES)}")

Codes racines valides    : 34663
Codes extensions valides : 16842


## 3. Reconstruction de la structure détaillée depuis le cache

Pour chaque code racine, on reconstruit sa structure depuis le cache API :
- Le code + libellé du racine
- Pour chaque axe : nom, règle (NotAllowed / AllowAlways), et liste des valeurs autorisées avec libellés.

In [6]:
def get_extensions_depuis_cache(scale_entities: list) -> list:
    """
    Descend récursivement dans le cache pour trouver tous les codes d'extension
    terminaux accessibles depuis une liste d'URI de départ.
    Ne garde que les codes du chapitre X (codes d'extension, premier caractère = 'X').
    """
    resultats = []
    a_traiter = list(scale_entities)
    vus = set()

    while a_traiter:
        uri = a_traiter.pop()
        if uri in vus:
            # URI déjà visitée : on évite les boucles infinies
            continue
        vus.add(uri)

        data = cache_url.get(uri)
        if not data:
            # URI absente du cache (non explorée précédemment) : on ignore
            continue

        code = data.get('code', '')
        if code and code.startswith('X'):  # MODIF : ne garder que le chapitre X
            # Le libellé vient d'abord du CSV de référence, sinon du cache API
            libelle = dict_libelle_ext.get(code, data.get('title', {}).get('@value', ''))
            resultats.append({'code': code, 'libelle': libelle})

        # On empile les enfants pour continuer la descente récursive
        for child_uri in data.get('child', []):
            a_traiter.append(child_uri.replace('http://', 'https://'))

    return resultats


def reconstruire_axes_detailles(code: str, uri: str) -> dict:
    """
    Reconstruit, pour un code racine donné, la liste de ses axes de post-coordination
    avec leurs règles (NotAllowed / AllowAlways) et leurs valeurs autorisées.
    """
    data = cache_url.get(uri)
    if not data:
        return None

    axes = []
    for axe in data.get('postcoordinationScale', []):
        axe_nom = axe.get('axisName', '').split('/')[-1]

        # Les axes hasAlternative* sont exclus (cf. notes de projet)
        if axe_nom.startswith('hasAlternative'):
            continue

        scale_entities = [e.replace('http://', 'https://') for e in axe.get('scaleEntity', [])]
        valeurs = get_extensions_depuis_cache(scale_entities)
        allow_multiple = axe.get('allowMultipleValues', 'NotAllowed')

        if valeurs:
            axes.append({
                'axe_nom':        axe_nom,
                'allow_multiple': allow_multiple,
                'valeurs':        valeurs,
            })

    if not axes:
        # Code racine sans axe exploitable : on ne le garde pas
        return None

    return {
        'code':    code,
        'libelle': dict_libelle_racine.get(code, ''),
        'axes':    axes,
    }

In [7]:
print("--- Exemple de get_extensions_depuis_cache ---")
# On va prendre un exemple pour le CODE_TEST utilisé précédemment '9B71.0Z'
CODE_EXAMPLE = '9B71.0Z'

# 1. Obtenir l'URI du code racine à partir du dictionnaire préchargé
uri_example = dict_uri.get(CODE_EXAMPLE)

if uri_example and uri_example in cache_url:
    # 2. Récupérer les données brutes du cache pour cette URI
    raw_data = cache_url.get(uri_example)

    # 3. Extraire une liste de scale_entities pour un axe de post-coordination (le premier, par exemple)
    scale_entities_for_axis = []
    if raw_data and 'postcoordinationScale' in raw_data and raw_data['postcoordinationScale']:
        # Prenons le premier axe pour cet exemple
        first_axis = raw_data['postcoordinationScale'][0]
        if 'scaleEntity' in first_axis:
            scale_entities_for_axis = [e.replace('http://', 'https://') for e in first_axis['scaleEntity']]

            print(f"Code racine d'exemple: {CODE_EXAMPLE}")
            print(f"URI d'exemple: {uri_example}")
            print(f"Liste des URIs de départ pour le premier axe (premiers 5) :\n  {scale_entities_for_axis[:5]}...")

            # 4. Appeler get_extensions_depuis_cache avec ces URIs
            example_extensions = get_extensions_depuis_cache(scale_entities_for_axis)

            print(f"\nRésultat de get_extensions_depuis_cache (premiers 5) :")
            for ext in example_extensions[:5]:
                print(f"  Code: {ext['code']}, Libellé: {ext['libelle']}")
            if len(example_extensions) > 5:
                print(f"  ... et {len(example_extensions) - 5} autres extensions.")
    else:
        print(f"Aucun axe de post-coordination trouvé pour le code {CODE_EXAMPLE} dans le cache.")
else:
    print(f"Le code {CODE_EXAMPLE} ou son URI n'est pas trouvé dans les données chargées.")

--- Exemple de get_extensions_depuis_cache ---
Code racine d'exemple: 9B71.0Z
URI d'exemple: http://id.who.int/icd/release/11/2024-01/mms/1006882070/unspecified
Liste des URIs de départ pour le premier axe (premiers 5) :
  ['https://id.who.int/icd/release/11/2024-01/mms/627678743', 'https://id.who.int/icd/release/11/2024-01/mms/271422288', 'https://id.who.int/icd/release/11/2024-01/mms/876572005', 'https://id.who.int/icd/release/11/2024-01/mms/1038788978']...

Résultat de get_extensions_depuis_cache (premiers 5) :
  Code: XK70, Libellé: Unilatéral, sans précision
  Code: XK9K, Libellé: Droit
  Code: XK8G, Libellé: Gauche
  Code: XK9J, Libellé: Bilatéral


In [8]:
# Reconstruction de la structure détaillée pour tous les codes racines ayant des axes
codes_avec_axes = df_axes['code'].unique()
# dict_uri = dict(zip(df_codes['code'], df_codes['uri'])) # Removed from here, moved to FOCMD6L-8ASr

axes_detailles = {}
codes_sans_structure = []

for code in tqdm(codes_avec_axes, desc="Reconstruction", unit="code"):
    uri = dict_uri.get(code)
    if not uri:
        codes_sans_structure.append(code)
        continue
    resultat = reconstruire_axes_detailles(code, uri)
    if resultat:
        axes_detailles[code] = resultat
    else:
        codes_sans_structure.append(code)

print(f"\nReconstruction terminée :")
print(f"  Codes avec structure détaillée : {len(axes_detailles)}")
print(f"  Codes sans structure           : {len(codes_sans_structure)}")

Reconstruction:   0%|          | 0/10287 [00:00<?, ?code/s]


Reconstruction terminée :
  Codes avec structure détaillée : 7435
  Codes sans structure           : 2852


## 4. Construction du prompt Mistral — Version renforcée

**Améliorations V2** :
- Instructions plus strictes sur l'utilisation exclusive des codes fournis
- Explicitation des règles par axe (NotAllowed / AllowAlways)
- Rappel explicite du format `racine&extension`
- Contrôle du réalisme médical renforcé

**Important** : on ne demande à Mistral QUE les codes post-coordonnés (format `racine&ext1&ext2...`), jamais les libellés en sortie. Les libellés sont reconstruits localement en post-traitement (section 5) à partir des CSV de référence, pour éviter tout risque d'hallucination.

In [9]:
SYSTEM_PROMPT = """Tu es un expert en codage médical avec la nomenclature CIM-11 (Classification Internationale des Maladies, 11ème révision) de l'OMS.

Ta mission : pour un code racine donné et la liste de ses axes de post-coordination, générer EXACTEMENT jusqu'à 100 codes post-coordonnés cliniquement réalistes.

RÈGLES STRICTES DE POST-COORDINATION CIM-11 :
1. Un code post-coordonné s'écrit : code_racine&extension1&extension2&...
2. Le caractère "&" sépare le code racine et chaque extension
3. Pour un axe "NotAllowed" (une seule valeur autorisée) : tu choisis AU MAXIMUM UNE valeur de cet axe
4. Pour un axe "AllowAlways" (plusieurs valeurs autorisées) : tu peux combiner plusieurs valeurs de cet axe
5. Tu n'es pas obligé d'utiliser tous les axes : un code post-coordonné peut n'utiliser qu'un sous-ensemble d'axes
6. INTERDICTION ABSOLUE : tu ne peux utiliser QUE les codes d'extension EXPLICITEMENT fournis dans la liste des valeurs autorisées de chaque axe. N'invente JAMAIS de code d'extension.
7. Chaque extension utilisée DOIT respecter l'axe auquel elle appartient (ne pas mélanger les extensions de différents axes en dehors des règles ci-dessus)

CRITÈRES DE RÉALISME MÉDICAL :
- Privilégie les combinaisons cliniquement COHÉRENTES (correspondance anatomique, physiopathologique)
- Privilégie les combinaisons FRÉQUENTES dans la pratique médicale
- ÉLIMINE les combinaisons anatomiquement ou physiologiquement impossibles
- Diversifie les combinaisons (différents axes, différentes valeurs) pour couvrir la variété clinique

FORMAT DE SORTIE STRICT :
Tu réponds UNIQUEMENT avec un bloc JSON, sans aucun texte avant ou après.
Structure : {"codes_postcoord": ["code1", "code2", ...]}
"""


def construire_user_prompt(info_code: dict) -> str:
    """
    Construit le prompt utilisateur listant le code racine, son libellé,
    et pour chaque axe la règle et les codes d'extension autorisés (avec libellés).
    """
    code = info_code['code']
    libelle = info_code['libelle']

    lignes = [
        f"CODE RACINE À POST-COORDONNER",
        f"Code    : {code}",
        f"Libellé : {libelle}",
        f"",
        f"AXES DE POST-COORDINATION DISPONIBLES ({len(info_code['axes'])}) :",
        f"",
    ]

    for i, axe in enumerate(info_code['axes'], start=1):
        regle = "NotAllowed = une seule valeur autorisée" if axe['allow_multiple'] == 'NotAllowed' else "AllowAlways = plusieurs valeurs autorisées simultanément"
        lignes.append(f"AXE {i} — {axe['axe_nom']}")
        lignes.append(f"  Règle : {regle}")
        lignes.append(f"  Codes d'extension autorisés ({len(axe['valeurs'])}) :")
        # On liste les libellés pour que Mistral choisisse des combinaisons médicalement cohérentes
        for v in axe['valeurs'][:100]:
            lignes.append(f"    - {v['code']} : {v['libelle']}")
        if len(axe['valeurs']) > 100:
            lignes.append(f"    ... ({len(axe['valeurs']) - 100} autres codes non listés)")
        lignes.append("")

    lignes.append(f"TÂCHE : génère jusqu'à 100 codes post-coordonnés RÉALISTES pour ce code racine.")
    lignes.append(f"CONTRAINTE : utilise UNIQUEMENT les codes d'extension listés ci-dessus.")
    lignes.append(f"Format de sortie : un JSON avec une clé 'codes_postcoord' contenant la liste des codes.")

    return "\n".join(lignes)


# Préfixe forcé pour orienter directement Mistral vers une réponse JSON
PREFIX_REPONSE = '{"codes_postcoord": ['


## 5. Extraction des codes et reconstruction des libellés

Deux fonctions utilisées partout dans la suite du notebook (test, export final) :
- `parser_reponse_mistral` : extrait les codes post-coordonnés de la réponse brute de Mistral, via regex (indépendant de la validité du JSON)
- `obtenir_libelles_postcoord` : reconstruit le libellé du code racine et les libellés de chaque extension, à partir des dictionnaires déjà chargés en section 2 (aucun appel API, aucune dépendance à Mistral)

In [10]:
def parser_reponse_mistral(contenu: str) -> list:
    """
    Extrait tous les codes post-coordonnés au format `racine&extension...` via regex.
    Indépendant de la validité du JSON.
    """
    pattern = r'[A-Za-z0-9][A-Za-z0-9.]*(?:&[A-Za-z0-9=.]+)+'
    codes = re.findall(pattern, contenu)
    return codes

### Validation des codes renvoyés par Mistral

`valider_code_cim11` découpe un code sur `&` :
- le premier segment doit être un **code racine** connu ;
- chaque segment suivant doit être un **code d'extension** connu (on accepte aussi un code racine,
  car la post-coordination OMS autorise l'association de deux entités, ex : `1A09.Y&1G41`).

Retourne `(est_valide, [codes_inconnus])`.

`filtrer_codes_valides` applique cette validation à une liste de codes et sépare
les codes réels des codes hallucinés, en journalisant ces derniers.

In [11]:
def valider_code_cim11(code):
    """
    Vérifie qu'un code (post-coordonné ou non) n'utilise que des codes réels.
    Retourne (est_valide, [codes_inconnus]).
    """
    parties = str(code).strip().split('&')
    racine, extensions = parties[0], parties[1:]

    inconnus = []
    if racine not in CODES_RACINES_VALIDES:
        inconnus.append(racine)

    for ext in extensions:
        # Une extension peut être un code X... ou un code racine (association d'entités)
        if ext not in CODES_EXTENSIONS_VALIDES and ext not in CODES_RACINES_VALIDES:
            inconnus.append(ext)

    return (len(inconnus) == 0, inconnus)


def filtrer_codes_valides(codes, contexte='', verbose=False):
    """
    Filtre une liste de codes renvoyés par Mistral.
    Retourne (codes_valides, rejetes) où rejetes est une liste de (code, [codes_inconnus]).
    """
    valides, rejetes = [], []
    for c in codes:
        ok, inconnus = valider_code_cim11(c)
        if ok:
            valides.append(c)
        else:
            rejetes.append((c, inconnus))

    if verbose and rejetes:
        print(f"[HALLUCINATION] {contexte} — {len(rejetes)} code(s) rejeté(s) :")
        for c, inconnus in rejetes[:10]:
            print(f"   {c}  →  inconnu(s) : {', '.join(inconnus)}")

    return valides, rejetes

In [12]:
def obtenir_libelles_postcoord(code_postcoord: str, code_racine: str) -> dict:
    """
    Reconstruit les libellés d'un code post-coordonné.
    Format attendu : racine&ext1&ext2&...
    """
    segments = code_postcoord.split('&')
    extensions = segments[1:]  # tout sauf la racine

    libelle_racine = dict_libelle_racine.get(code_racine, '')

    libelles_extensions = []
    for ext in extensions:
        # certaines extensions peuvent contenir "=" (ex: valeur assignée) -> on ne garde que le code
        code_ext = ext.split('=')[0]
        libelle_ext = dict_libelle_ext.get(code_ext, '')
        libelles_extensions.append(f"{ext} : {libelle_ext}" if libelle_ext else ext)

    return {
        'libelle_racine': libelle_racine,
        'libelles_extensions': " | ".join(libelles_extensions),
    }

### Test manuel de la validation

Contrôle rapide sur trois cas : un code post-coordonné valide, un code contenant une
extension hallucinée (`XA13Z`), et une racine inexistante.

In [13]:
# Cellule de test — vérification du comportement de valider_code_cim11
for code_test_valid in ['9B71.0Z&XK9K&XS5W', '1A62.2&XA13Z', 'ZZZZ&XA6GV0']:
    print(code_test_valid, '→', valider_code_cim11(code_test_valid))

9B71.0Z&XK9K&XS5W → (True, [])
1A62.2&XA13Z → (False, ['XA13Z'])
ZZZZ&XA6GV0 → (False, ['ZZZZ'])


## 6. Test synchrone sur 1 code

In [14]:
CODE_TEST = '9B71.0Z'

if CODE_TEST not in axes_detailles:
    print(f"{CODE_TEST} non disponible")
else:
    info = axes_detailles[CODE_TEST]
    user_prompt = construire_user_prompt(info)

    print(f"User prompt (500 premiers caractères) :")
    print(user_prompt[:500] + "...")
    print()

    # Appel synchrone (hors batch) pour valider rapidement le prompt et le format de réponse
    response = client_mistral.chat.complete(
        model=MODEL_MISTRAL,
        messages=[
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": user_prompt},
            {"role": "assistant", "content": PREFIX_REPONSE, "prefix": True},
        ],
        max_tokens=8000,
    )

    contenu = response.choices[0].message.content
    print(f"Réponse Mistral (1000 premiers caractères) :")
    print(contenu[:1000])
    print()

    # Extraction des codes + reconstruction des libellés pour vérifier le rendu final
    codes_extraits_test = parser_reponse_mistral(contenu)
    print(f"Nombre de codes post-coordonnés extraits : {len(codes_extraits_test)}")
    print(f"Aperçu avec libellés (10 premiers) :")
    for code_pc in codes_extraits_test[:10]:
        libelles = obtenir_libelles_postcoord(code_pc, CODE_TEST)
        print(f"  {code_pc}")
        print(f"    Racine     : {libelles['libelle_racine']}")
        print(f"    Extensions : {libelles['libelles_extensions']}")

User prompt (500 premiers caractères) :
CODE RACINE À POST-COORDONNER
Code    : 9B71.0Z
Libellé : Rétinopathie diabétique, sans précision

AXES DE POST-COORDINATION DISPONIBLES (2) :

AXE 1 — laterality
  Règle : NotAllowed = une seule valeur autorisée
  Codes d'extension autorisés (4) :
    - XK70 : Unilatéral, sans précision
    - XK9K : Droit
    - XK8G : Gauche
    - XK9J : Bilatéral

AXE 2 — hasSeverity
  Règle : NotAllowed = une seule valeur autorisée
  Codes d'extension autorisés (3) :
    - XS25 : Sévère
    - XS0T : Modéré
  ...

Réponse Mistral (1000 premiers caractères) :
{"codes_postcoord": [ "9B71.0Z&XK70&XS25", "9B71.0Z&XK9K&XS25", "9B71.0Z&XK8G&XS25", "9B71.0Z&XK9J&XS25", "9B71.0Z&XK70&XS0T", "9B71.0Z&XK9K&XS0T", "9B71.0Z&XK8G&XS0T", "9B71.0Z&XK9J&XS0T", "9B71.0Z&XK70&XS5W", "9B71.0Z&XK9K&XS5W", "9B71.0Z&XK8G&XS5W", "9B71.0Z&XK9J&XS5W", "9B71.0Z&XK70", "9B71.0Z&XK9K", "9B71.0Z&XK8G", "9B71.0Z&XK9J", "9B71.0Z&XS25", "9B71.0Z&XS0T", "9B71.0Z&XS5W", "9B71.0Z&XK9K&XS25", "9B

## 7. Préparation des prompts pour tous les codes

In [15]:
# Un prompt système + utilisateur par code racine, prêt à être envoyé en batch
lignes_prompts = []
for code, info in axes_detailles.items():
    lignes_prompts.append({
        'code':          code,
        'system_prompt': SYSTEM_PROMPT,
        'user_prompt':   construire_user_prompt(info),
        'prefix':        PREFIX_REPONSE,
    })

df_prompts = pd.DataFrame(lignes_prompts)
n_batches = (len(df_prompts) + N_CODES_PAR_BATCH - 1) // N_CODES_PAR_BATCH

print(f"Codes à traiter : {len(df_prompts)}")
print(f"Nombre de batches : {n_batches}")

Codes à traiter : 7435
Nombre de batches : 75


## 8. Lancement des batches Mistral

**Reprise après interruption** : si un fichier `batch_NNNN.jsonl` existe déjà dans V2/batches, on le saute.

**Robustesse** : `run_batch_job` réessaie automatiquement en cas d'erreur transitoire de l'API pendant le polling du statut (ex : erreur 404 alors que le job existe bien côté serveur).

In [16]:
def create_input_file_codes(client, df_batch):
    """Construit et upload le fichier JSONL d'entrée pour un batch de codes racines."""
    buffer = BytesIO()
    for _, ligne in df_batch.iterrows():
        request = {
            "custom_id": ligne['code'],
            "body": {
                "max_tokens": 8000,
                "messages": [
                    {"role": "system",    "content": ligne['system_prompt']},
                    {"role": "user",      "content": ligne['user_prompt']},
                    {"role": "assistant", "content": ligne['prefix'], "prefix": True},
                ],
            },
        }
        buffer.write(json.dumps(request, ensure_ascii=False).encode("utf-8"))
        buffer.write("\n".encode("utf-8"))
    return client.files.upload(
        file=File(file_name="codes_postcoord.jsonl", content=buffer.getvalue()),
        purpose="batch",
    )


def run_batch_job(client, input_file, model, max_retries=5):
    """
    Lance un batch et attend sa complétion en pollant le statut.
    Réessaie jusqu'à `max_retries` fois en cas d'erreur transitoire de l'API (ex : 404).
    """
    jobs_api = client.batch.jobs
    batch_job = jobs_api.create(
        input_files=[input_file.id],
        model=model,
        endpoint="/v1/chat/completions",
        metadata={"job_type": "verif_postcoord_v2"},
    )
    while batch_job.status in ["QUEUED", "RUNNING"]:
        for tentative in range(max_retries):
            try:
                batch_job = jobs_api.get(job_id=batch_job.id)
                break
            except Exception as e:
                print(f"  Erreur de polling (tentative {tentative+1}/{max_retries}) : {e}")
                time.sleep(10)
        else:
            raise RuntimeError(f"Échec du polling après {max_retries} tentatives pour le job {batch_job.id}")
        time.sleep(5)
    print(f"  Batch {batch_job.id} — statut : {batch_job.status}")
    return batch_job


def download_file(client, file_id, output_path):
    """Télécharge le fichier de résultat d'un batch vers le Drive."""
    if file_id is not None:
        output_file = client.files.download(file_id=file_id)
        with open(output_path, "wb") as f:
            for chunk in output_file.stream:
                f.write(chunk)

In [17]:
# Boucle principale : un batch de 100 codes racines à la fois, avec reprise automatique
for i in range(n_batches):
    debut_idx = i * N_CODES_PAR_BATCH
    fin_idx   = min((i + 1) * N_CODES_PAR_BATCH, len(df_prompts))
    output_path = os.path.join(PATH_BATCH_DIR, f"batch_{i:04d}.jsonl")

    if os.path.exists(output_path):
        # Batch déjà traité lors d'une exécution précédente : on ne le relance pas
        print(f"Batch {i+1}/{n_batches} déjà fait (skip)")
        continue

    print(f"Batch {i+1}/{n_batches} — codes {debut_idx} à {fin_idx-1}")
    df_batch = df_prompts.iloc[debut_idx:fin_idx].reset_index(drop=True)

    input_file = create_input_file_codes(client_mistral, df_batch)
    print(f"  Fichier uploadé : {input_file.id}")

    batch_job = run_batch_job(client_mistral, input_file, MODEL_MISTRAL)

    download_file(client_mistral, batch_job.output_file, output_path)
    print(f"  Résultat → {output_path}")
    print()

print(f"Tous les batches terminés ({n_batches} batches)")

Batch 1/75 — codes 0 à 99
  Fichier uploadé : 2b3dadb2-7340-4efc-ba41-8a6e3de657a3
  Batch 0a456a76-2947-4a27-8c79-cd992328b80a — statut : SUCCESS
  Résultat → /content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/verif_postcoord/batches/batch_0000.jsonl

Batch 2/75 — codes 100 à 199
  Fichier uploadé : ae950a95-55e0-42fb-8d59-4a922fd2f555
  Batch 1d27fb75-d7ab-467c-8846-20936fce5b67 — statut : SUCCESS
  Résultat → /content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/verif_postcoord/batches/batch_0001.jsonl

Batch 3/75 — codes 200 à 299
  Fichier uploadé : a131d739-a2a3-4428-9894-49643f9c66ba
  Batch 92adba89-b3c2-4633-a915-db26c00864ac — statut : SUCCESS
  Résultat → /content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/verif_postcoord/batches/batch_0002.jsonl

Batch 4/75 — codes 300 à 399
  Fichier uploadé : a7ca240f-2d93-4558-b07f-2294bebdec87
  Batch 23638dfe-e9e2-4747-acbc-8485fa0beddb — statut : SUCCESS
  Résultat → /content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/verif_postcoo

## 9. Parsing des résultats et export final

Pour chaque code post-coordonné extrait, on reconstruit le libellé du code racine et les libellés de chaque extension via `obtenir_libelles_postcoord` (section 5), sans nouvel appel à Mistral.

In [18]:
lignes_resultat = []
erreurs = []
hallucinations = []   # (code_racine, code_postcoord, [codes_inconnus])

fichiers_batch = sorted([
    f for f in os.listdir(PATH_BATCH_DIR)
    if f.startswith('batch_') and f.endswith('.jsonl')
])

print(f"Parsing de {len(fichiers_batch)} fichiers de batch...")

for fichier in tqdm(fichiers_batch, desc="Parsing", unit="fichier"):
    path = os.path.join(PATH_BATCH_DIR, fichier)
    with open(path, 'r', encoding='utf-8') as f:
        for ligne in f:
            try:
                resultat = json.loads(ligne.strip())
            except json.JSONDecodeError:
                # Ligne corrompue : on l'ignore
                continue

            code_racine = resultat.get('custom_id', '')

            try:
                contenu = resultat['response']['body']['choices'][0]['message']['content']
            except (KeyError, IndexError, TypeError):
                erreurs.append((code_racine, "réponse vide ou malformée"))
                continue

            codes_extraits = parser_reponse_mistral(contenu)
            if not codes_extraits:
                erreurs.append((code_racine, "aucun code extrait"))
                continue

            # Filtrage anti-hallucination : on écarte les codes inexistants
            # (Mistral invente parfois des extensions, ex : XA13Z)
            codes_extraits, codes_rejetes = filtrer_codes_valides(codes_extraits, code_racine)
            for code_rej, inconnus in codes_rejetes:
                hallucinations.append((code_racine, code_rej, inconnus))
            if not codes_extraits:
                erreurs.append((code_racine, "tous les codes hallucinés"))
                continue

            for code_pc in codes_extraits:
                # Reconstruction des libellés (racine + extensions) en local, sans appel API
                libelles = obtenir_libelles_postcoord(code_pc, code_racine)
                lignes_resultat.append({
                    'code_racine':          code_racine,
                    'libelle_racine':       libelles['libelle_racine'],
                    'code_postcoord':       code_pc,
                    'libelles_extensions':  libelles['libelles_extensions'],
                })

df_resultat = pd.DataFrame(lignes_resultat)
df_resultat.to_csv(PATH_RESULTAT, index=False, encoding='utf-8-sig')

print(f"\nFichier exporté : {PATH_RESULTAT}")
print(f"  Nombre de lignes (codes post-coord)   : {len(df_resultat)}")
if len(df_resultat) > 0:
    print(f"  Nombre de codes racines couverts      : {df_resultat['code_racine'].nunique()}")
    print(f"  Moyenne de codes post-coord par code  : {len(df_resultat) / df_resultat['code_racine'].nunique():.1f}")
print(f"  Codes hallucinés écartés              : {len(hallucinations)}")
print()
if hallucinations:
    print("Aperçu des codes hallucinés (10 premiers) :")
    for racine, code_h, inconnus in hallucinations[:10]:
        print(f"    {racine} : {code_h} → inconnu(s) : {', '.join(inconnus)}")
    print()
if erreurs:
    print(f"Erreurs rencontrées : {len(erreurs)}")
    for code, msg in erreurs[:10]:
        print(f"    {code} : {msg}")

Parsing de 75 fichiers de batch...


Parsing:   0%|          | 0/75 [00:00<?, ?fichier/s]


Fichier exporté : /content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/verif_postcoord/data/codes_postcoord_realistes_v2.csv
  Nombre de lignes (codes post-coord)   : 570132
  Nombre de codes racines couverts      : 7431
  Moyenne de codes post-coord par code  : 76.7
  Codes hallucinés écartés              : 21894

Aperçu des codes hallucinés (10 premiers) :
    1A03 : 1A03&Escherichia → inconnu(s) : Escherichia
    1A1Y : 1A1Y&XN6AS → inconnu(s) : XN6AS
    1A36.1Y : 1A36.1Y&XA99N3&XA8QA8&XA3DM0 → inconnu(s) : XA3DM0
    1C41 : 1C41&XN2QM&XN9ZG&XN1V6 → inconnu(s) : XN1V6
    1C41 : 1C41&XN2QM&XN9ZG&XN5J9 → inconnu(s) : XN5J9
    1C41 : 1C41&XN9ZG&XN1V6 → inconnu(s) : XN1V6
    1C41 : 1C41&XN9ZG&XN5J9 → inconnu(s) : XN5J9
    1C41 : 1C41&XN9ZG&XN1V6&XN5J9 → inconnu(s) : XN1V6, XN5J9
    1C41 : 1C41&XN9ZG&XN1V6&XN4N7 → inconnu(s) : XN1V6
    1C41 : 1C41&XN9ZG&XN1V6&XN5J9&XN4N7 → inconnu(s) : XN1V6, XN5J9

Erreurs rencontrées : 4
    2B72 : tous les codes hallucinés
    2C7Z : tous les 

### Export du journal des hallucinations

On sauvegarde la liste complète des codes écartés dans `codes_hallucines_v2.csv`
(colonnes : `code_racine`, `code_postcoord`, `codes_inconnus`). Ce fichier permet
de quantifier le taux d'hallucination de Mistral et d'identifier les codes racines
les plus problématiques.

In [19]:
# Export du journal des codes hallucinés
PATH_HALLUCINATIONS = PATH_VERIF + "data/codes_hallucines_v2.csv"

df_hallucinations = pd.DataFrame(
    [{'code_racine': r, 'code_postcoord': c, 'codes_inconnus': '|'.join(inc)}
     for r, c, inc in hallucinations]
)
df_hallucinations.to_csv(PATH_HALLUCINATIONS, index=False, encoding='utf-8-sig')

print(f"Fichier exporté : {PATH_HALLUCINATIONS}")
print(f"  Codes hallucinés : {len(df_hallucinations)}")

if len(df_hallucinations) > 0:
    total = len(df_hallucinations) + len(df_resultat)
    print(f"  Taux d'hallucination : {100 * len(df_hallucinations) / total:.2f} %")
    print()
    print("Codes d'extension inventés les plus fréquents :")
    print(
        df_hallucinations['codes_inconnus']
        .str.split('|').explode()
        .value_counts().head(15).to_string()
    )

Fichier exporté : /content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/verif_postcoord/data/codes_hallucines_v2.csv
  Codes hallucinés : 21894
  Taux d'hallucination : 3.70 %

Codes d'extension inventés les plus fréquents :
codes_inconnus
XS2P       2670
XS3H       2361
XS2H       1417
XS3P       1139
XS4R        823
XN0J9       273
XA7T        152
XA1AZ       143
XA1V94      138
XA1V98      116
ME84.21     110
ME84.4      102
MEDICAL     101
ME84.5       98
XA1Y         94


In [20]:
import pandas as pd

df_postcoord_v2 = pd.read_csv(
    '/content/drive/MyDrive/Colab_Notebooks/Serenic_M/api/verif_postcoord/data/codes_postcoord_realistes_v2.csv'
)
print(df_postcoord_v2.shape)
print(df_postcoord_v2.columns.tolist())
print(df_postcoord_v2.head(5))
print(f"Codes racines uniques : {df_postcoord_v2['code_racine'].nunique()}")

(570132, 4)
['code_racine', 'libelle_racine', 'code_postcoord', 'libelles_extensions']
  code_racine                                     libelle_racine  \
0      1A62.2  Syphilis tardive symptomatique d'autres locali...   
1      1A62.2  Syphilis tardive symptomatique d'autres locali...   
2      1A62.2  Syphilis tardive symptomatique d'autres locali...   
3      1A62.2  Syphilis tardive symptomatique d'autres locali...   
4      1A62.2  Syphilis tardive symptomatique d'autres locali...   

         code_postcoord                                libelles_extensions  
0         1A62.2&XA6GV0                                   XA6GV0 : Abdomen  
1         1A62.2&XA3KX0                          XA3KX0 : Paroi abdominale  
2         1A62.2&XA4SN6               XA4SN6 : Paroi abdominale antérieure  
3  1A62.2&XA4TC0&XA0NH8  XA4TC0 : Partie inférieure de l'abdomen | XA0N...  
4         1A62.2&XA6N20                              XA6N20 : Hypogastrium  
Codes racines uniques : 7431
